In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("data/processed/ottawa_places_cleaned_v2.csv")
df = pd.read_csv(csv_path)

# Ajouter les colonnes si elles n'existent pas
if "google_rating" not in df.columns:
    df["google_rating"] = pd.NA

if "google_user_rating_count" not in df.columns:
    df["google_user_rating_count"] = pd.NA

if "rating_source" not in df.columns:
    df["rating_source"] = pd.NA

# Sauvegarder
df.to_csv(csv_path, index=False)

print("Colonnes ajoutées ")
print(df.columns.tolist())
display(df.head())

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

TypeError: 'function' object does not support item assignment

In [ ]:
import os
import requests
import pandas as pd

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
print("Clé chargée :", API_KEY[:10] + "..." if API_KEY else "Aucune")

url = "https://places.googleapis.com/v1/places:searchText"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount"
}
payload = {
    "textQuery": "Lord Elgin Ottawa",
    "languageCode": "fr",
    "regionCode": "CA",
    "maxResultCount": 3
}

r = requests.post(url, headers=headers, json=payload, timeout=30)

print("Status code :", r.status_code)
print(r.text[:1500])

In [ ]:
# Colonnes numériques
for col in ["google_rating", "google_user_rating_count", "google_match_score"]:
    if col not in df.columns:
        df[col] = pd.NA
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Colonnes texte
for col in ["rating_source", "google_place_id"]:
    if col not in df.columns:
        df[col] = pd.NA
    df[col] = df[col].astype("string")

In [ ]:
import os
import math
import time
import difflib
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
if not API_KEY:
    raise ValueError("Clé API absente. Définis GOOGLE_MAPS_API_KEY avant de lancer ce script.")

csv_path = Path("data/processed/ottawa_places_cleaned_v2.csv")
df = pd.read_csv(csv_path)

# Initialisation propre des colonnes
for col in ["google_rating", "google_user_rating_count", "google_match_score"]:
    if col not in df.columns:
        df[col] = pd.NA
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["rating_source", "google_place_id"]:
    if col not in df.columns:
        df[col] = pd.NA
    df[col] = df[col].astype("string")

TEXT_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"
HEADERS = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": ",".join([
        "places.id",
        "places.displayName",
        "places.formattedAddress",
        "places.location",
        "places.rating",
        "places.userRatingCount",
    ])
}

def safe_text(x):
    if pd.isna(x):
        return ""
    v = str(x).strip()
    if v.lower() in {"nan", "none", "null", "undefined"}:
        return ""
    return v

def haversine_km(lat1, lon1, lat2, lon2):
    if any(pd.isna(v) for v in [lat1, lon1, lat2, lon2]):
        return None
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def similarity(a, b):
    a = safe_text(a).lower()
    b = safe_text(b).lower()
    if not a or not b:
        return 0.0
    return difflib.SequenceMatcher(None, a, b).ratio()

def build_query(row):
    name = safe_text(row.get("name"))
    address = safe_text(row.get("address_clean")) or safe_text(row.get("address"))
    city = safe_text(row.get("addr_city")) or "Ottawa"
    return ", ".join([x for x in [name, address, city, "ON", "Canada"] if x])

def google_text_search(query, max_results=5):
    payload = {
        "textQuery": query,
        "languageCode": "fr",
        "regionCode": "CA",
        "maxResultCount": max_results,
    }
    r = requests.post(TEXT_SEARCH_URL, headers=HEADERS, json=payload, timeout=30)
    r.raise_for_status()
    return r.json().get("places", [])

def pick_best_match(row, places):
    row_name = safe_text(row.get("name"))
    row_addr = safe_text(row.get("address_clean")) or safe_text(row.get("address"))
    row_lat = pd.to_numeric(row.get("lat"), errors="coerce")
    row_lon = pd.to_numeric(row.get("lon"), errors="coerce")

    best = None
    best_score = -1

    for p in places:
        p_name = safe_text((p.get("displayName") or {}).get("text"))
        p_addr = safe_text(p.get("formattedAddress"))
        p_loc = p.get("location") or {}
        p_lat = p_loc.get("latitude")
        p_lon = p_loc.get("longitude")

        name_score = similarity(row_name, p_name)
        addr_score = similarity(row_addr, p_addr)

        dist_score = 0.0
        dist_km = haversine_km(row_lat, row_lon, p_lat, p_lon)
        if dist_km is not None:
            if dist_km <= 0.2:
                dist_score = 1.0
            elif dist_km <= 1.0:
                dist_score = 0.8
            elif dist_km <= 3.0:
                dist_score = 0.5
            else:
                dist_score = 0.1

        score = 0.6 * name_score + 0.25 * addr_score + 0.15 * dist_score

        if score > best_score:
            best_score = score
            best = {
                "place_id": p.get("id"),
                "display_name": p_name,
                "formatted_address": p_addr,
                "rating": p.get("rating"),
                "user_rating_count": p.get("userRatingCount"),
                "match_score": round(score, 4),
                "distance_km": None if dist_km is None else round(dist_km, 3),
            }

    return best

# Mode complet
TEST_MODE = False
TEST_N = 10

work_df = df.copy()
if TEST_MODE:
    work_df = work_df.head(TEST_N).copy()

for idx, row in tqdm(work_df.iterrows(), total=len(work_df)):
    # saute si déjà enrichi
    if pd.notna(df.at[idx, "google_rating"]) and pd.notna(df.at[idx, "google_user_rating_count"]):
        continue

    query = build_query(row)

    try:
        places = google_text_search(query, max_results=5)
        if not places:
            continue

        best = pick_best_match(row, places)
        if not best:
            continue

        # seuil de confiance
        if best["match_score"] >= 0.72:
            df.at[idx, "google_place_id"] = best["place_id"]
            df.at[idx, "google_rating"] = best["rating"]
            df.at[idx, "google_user_rating_count"] = best["user_rating_count"]
            df.at[idx, "rating_source"] = "google_places_api"
            df.at[idx, "google_match_score"] = best["match_score"]

        time.sleep(0.15)  # petite pause pour rester raisonnable

    except Exception as e:
        print(f"[WARN] ligne {idx} | {row.get('name')} | {e}")

In [ ]:
# Sauvegarde dans un nouveau fichier
out_path = Path("data/processed/ottawa_places_enriched_google.csv")
df.to_csv(out_path, index=False)

print(f"Fichier enrichi sauvegardé : {out_path}")

In [ ]:
filled = df["google_rating"].notna().sum()
total = len(df)

print(f"Lignes enrichies avec une vraie note : {filled}/{total}")
print(f"Pourcentage : {filled/total:.2%}")

display(
    df.loc[df["google_rating"].notna(), 
           ["name", "google_rating", "google_user_rating_count", "rating_source", "google_match_score"]]
      .head(20)
)

# Ajout des photos

In [12]:
import pandas as pd
from pathlib import Path

csv_path = Path("data/processed/ottawa_places_enriched_google.csv")
df = pd.read_csv(csv_path)

# Ajouter les colonnes si elles n'existent pas
if "google_photo_name" not in df.columns:
    df["google_photo_name"] = pd.NA

if "google_photo_url" not in df.columns:
    df["google_photo_url"] = pd.NA

if "google_photo_attribution" not in df.columns:
    df["google_photo_attribution"] = pd.NA

if "google_photo_match_score" not in df.columns:
    df["google_photo_match_score"] = pd.NA

# Sauvegarder
df.to_csv(csv_path, index=False)

print("Colonnes ajoutées ")
print(df.columns.tolist())
display(df.head())

Colonnes ajoutées 
['osm_type', 'osm_id', 'name', 'place_type', 'amenity', 'tourism', 'cuisine', 'lat', 'lon', 'addr_housenumber', 'addr_street', 'addr_city', 'addr_postcode', 'address', 'phone', 'website', 'opening_hours', 'wheelchair', 'brand', 'source', 'tags_json', 'address_clean', 'dist_to_center_km', 'text', 'eval_label', 'google_rating', 'google_user_rating_count', 'rating_source', 'google_match_score', 'google_place_id', 'google_photo_name', 'google_photo_url', 'google_photo_attribution', 'google_photo_match_score']


,osm_type,osm_id,name,place_type,amenity,tourism,cuisine,lat,lon,addr_housenumber,...,eval_label,google_rating,google_user_rating_count,rating_source,google_match_score,google_place_id,google_photo_name,google_photo_url,google_photo_attribution,google_photo_match_score
0,relation,14526702,Les Suites Hotel,hotel,Unknown,hotel,Unknown,45.426134,-75.688731,130,...,hotel,4.0,2249.0,google_places_api,0.8637,ChIJpXHm3wMFzkwRLC22CEvfRYg,NaN,NaN,NaN,<NA>
1,way,31813259,Embassy Hotel and Suites,hotel,Unknown,hotel,Unknown,45.419679,-75.688818,25,...,hotel,4.0,1277.0,google_places_api,0.8933,ChIJJSkEpKkFzkwRgKmEk0gQEA8,NaN,NaN,NaN,<NA>
2,way,31813278,The Business Inn & Suites,hotel,Unknown,hotel,Unknown,45.416980,-75.689237,180,...,hotel,4.4,2778.0,google_places_api,0.9429,ChIJOViXzBYPzkwRwgwgubhv0Lk,NaN,NaN,NaN,<NA>
3,way,46016942,Ottawa Jail Hostel,hostel,Unknown,hostel,Unknown,45.425111,-75.688451,75,...,hostel,4.3,2395.0,google_places_api,0.8394,ChIJQ4GyNgEFzkwRdukc-ESL2IA,NaN,NaN,NaN,<NA>
4,way,55411730,Sonder Rideau Apartments Downtown,hotel,Unknown,hotel,Unknown,45.420741,-75.694605,161,...,hotel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [ ]:
import os

os.environ["GOOGLE_MAPS_API_KEY"] = "AIza.....ks"

print(os.getenv("GOOGLE_MAPS_API_KEY"))

In [10]:
import os
import requests

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

url = "https://places.googleapis.com/v1/places:searchText"
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.photos"
}
payload = {
    "textQuery": "Les Suites Hotel Ottawa",
    "languageCode": "fr",
    "regionCode": "CA",
    "maxResultCount": 3
}

r = requests.post(url, headers=headers, json=payload, timeout=30)
print("status:", r.status_code)
print(r.text[:1000])

status: 200
{
  "places": [
    {
      "id": "ChIJpXHm3wMFzkwRLC22CEvfRYg",
      "displayName": {
        "text": "Les Suites Hotel Ottawa",
        "languageCode": "en"
      },
      "photos": [
        {
          "name": "places/ChIJpXHm3wMFzkwRLC22CEvfRYg/photos/ATCDNfVUp0Mg9V1rcr9DqfR5ePaZOO16m9JjrXrMJCty9DZ6v288CYc45tjBwoG5ZZFkY1OdIZ_emt86BSNH7SaYC4Wy4kXK0qS0vB4XMBIL944infseb4xvXDW8_829l7L1n5R30uQs90i5-o0g9jzgVqmDOqn-G-88DjX8zFapC2dyWDaPJECy36CSUHbWScwSJng7lZNl_CypSoz3Jgs0RfJXNI4x5bJaQF6bJuze3PcqEDsfk5AY1BeduSsgCmBgVzC1qCQxbVB7hETXPy8KWYWBeT-NIcStY7bpBeuLYflVuQ",
          "widthPx": 1440,
          "heightPx": 965,
          "authorAttributions": [
            {
              "displayName": "Les Suites Hotel Ottawa",
              "uri": "https://maps.google.com/maps/contrib/105629125846103367135",
              "photoUri": "https://lh3.googleusercontent.com/a-/ALV-UjVbbgYzxvwAE9heAYAht9ajtPn5ORyYWGLYZxBJdMuY4Bi9aew=s100-p-k-no-mo"
            }
          ],
          "flagCo

In [ ]:
import os
import time
import math
import difflib
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
if not API_KEY:
    raise ValueError("Clé API absente. Définis GOOGLE_MAPS_API_KEY avant de lancer ce script.")

csv_path = Path("data/processed/ottawa_places_enriched_google.csv")
df = pd.read_csv(csv_path)

# Initialisation propre colonnes
# colonnes texte
for col in ["google_photo_url", "google_photo_name", "google_photo_attribution"]:
    if col not in df.columns:
        df[col] = pd.NA
    df[col] = df[col].astype("string")

# colonne numérique
if "google_photo_match_score" not in df.columns:
    df["google_photo_match_score"] = pd.NA
df["google_photo_match_score"] = pd.to_numeric(df["google_photo_match_score"], errors="coerce")

TEXT_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"

HEADERS = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": ",".join([
        "places.id",
        "places.displayName",
        "places.formattedAddress",
        "places.location",
        "places.photos",
    ])
}

# Utils
def safe_text(x):
    if pd.isna(x):
        return ""
    v = str(x).strip()
    if v.lower() in {"nan", "none", "null", "undefined", "<na>"}:
        return ""
    return v

def normalize_query_text(s):
    s = safe_text(s)
    s = s.replace("\n", " ").replace("\r", " ")
    s = " ".join(s.split())
    return s

def similarity(a, b):
    a = safe_text(a).lower()
    b = safe_text(b).lower()
    if not a or not b:
        return 0.0
    return difflib.SequenceMatcher(None, a, b).ratio()

def haversine_km(lat1, lon1, lat2, lon2):
    vals = [lat1, lon1, lat2, lon2]
    if any(pd.isna(v) for v in vals):
        return None
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

def build_queries(row):
    name = normalize_query_text(row.get("name"))
    address = normalize_query_text(row.get("address_clean")) or normalize_query_text(row.get("address"))
    city = normalize_query_text(row.get("addr_city")) or "Ottawa"

    queries = []

    # requête complète
    full_query = ", ".join([x for x in [name, address, city, "ON", "Canada"] if x])
    if full_query:
        queries.append(full_query)

    # fallback plus simple
    q2 = ", ".join([x for x in [name, city, "ON", "Canada"] if x])
    if q2 and q2 not in queries:
        queries.append(q2)

    # fallback minimal
    q3 = f"{name}, Ottawa"
    if q3 and q3 not in queries:
        queries.append(q3)

    return queries

def google_text_search(query, max_results=5):
    payload = {
        "textQuery": query,
        "languageCode": "fr",
        "regionCode": "CA",
        "maxResultCount": max_results,
    }
    r = requests.post(TEXT_SEARCH_URL, headers=HEADERS, json=payload, timeout=30)
    r.raise_for_status()
    return r.json().get("places", [])

def pick_best_match(row, places):
    row_name = safe_text(row.get("name"))
    row_addr = safe_text(row.get("address_clean")) or safe_text(row.get("address"))
    row_lat = pd.to_numeric(row.get("lat"), errors="coerce")
    row_lon = pd.to_numeric(row.get("lon"), errors="coerce")

    best = None
    best_score = -1

    for p in places:
        p_name = safe_text((p.get("displayName") or {}).get("text"))
        p_addr = safe_text(p.get("formattedAddress"))
        p_loc = p.get("location") or {}
        p_lat = p_loc.get("latitude")
        p_lon = p_loc.get("longitude")

        name_score = similarity(row_name, p_name)
        addr_score = similarity(row_addr, p_addr)

        dist_score = 0.0
        dist_km = haversine_km(row_lat, row_lon, p_lat, p_lon)
        if dist_km is not None:
            if dist_km <= 0.2:
                dist_score = 1.0
            elif dist_km <= 1.0:
                dist_score = 0.8
            elif dist_km <= 3.0:
                dist_score = 0.5
            else:
                dist_score = 0.1

        score = 0.6 * name_score + 0.25 * addr_score + 0.15 * dist_score

        photos = p.get("photos") or []
        first_photo = photos[0] if photos else {}
        attributions = first_photo.get("authorAttributions") or []

        if score > best_score:
            best_score = score
            best = {
                "place_id": p.get("id"),
                "match_score": round(score, 4),
                "photo_name": first_photo.get("name"),
                "photo_attribution": " | ".join(
                    [safe_text(a.get("displayName")) for a in attributions if safe_text(a.get("displayName"))]
                ) or pd.NA,
            }

    return best

def build_photo_url(photo_name, max_width=1000):
    if not photo_name or pd.isna(photo_name):
        return pd.NA
    # URL directe qui redirige vers l'image
    return f"https://places.googleapis.com/v1/{photo_name}/media?key={API_KEY}&maxWidthPx={max_width}"

# Mode complet
TEST_MODE = False
TEST_N = 10

work_df = df.copy()
if TEST_MODE:
    work_df = work_df.head(TEST_N).copy()

for idx, row in tqdm(work_df.iterrows(), total=len(work_df)):
    # skip si photo déjà présente
    if pd.notna(df.at[idx, "google_photo_url"]):
        continue

    queries = build_queries(row)
    best = None

    for query in queries:
        try:
            places = google_text_search(query, max_results=5)
            if not places:
                continue

            best = pick_best_match(row, places)
            if best:
                break

        except requests.HTTPError as e:
            # si une requête casse, on essaie le fallback suivant
            print(f"[WARN] ligne {idx} | {row.get('name')} | requête échouée: {query} | {e}")
            continue
        except Exception as e:
            print(f"[WARN] ligne {idx} | {row.get('name')} | {e}")
            continue

    if not best:
        continue

    # seuil de confiance
    if best["match_score"] >= 0.72 and pd.notna(best["photo_name"]):
        df.at[idx, "google_photo_name"] = best["photo_name"]
        df.at[idx, "google_photo_url"] = build_photo_url(best["photo_name"], max_width=1000)
        df.at[idx, "google_photo_attribution"] = best["photo_attribution"]
        df.at[idx, "google_photo_match_score"] = float(best["match_score"])

    time.sleep(0.12)

 13%|█▎        | 142/1104 [03:07<20:32,  1.28s/it]

[WARN] ligne 142 | Host Indian Cuisine | requête échouée: Host Indian Cuisine, 622 Montréal Road, Ottawa Unknown, Ottawa, ON, Canada | 400 Client Error: Bad Request for url: https://places.googleapis.com/v1/places:searchText


 22%|██▏       | 245/1104 [05:27<19:20,  1.35s/it]

[WARN] ligne 245 | Banditos | requête échouée: Banditos, 683 Bank Street, Ottawa Unknown, Ottawa, ON, Canada | 400 Client Error: Bad Request for url: https://places.googleapis.com/v1/places:searchText


 23%|██▎       | 254/1104 [05:41<20:49,  1.47s/it]

...

 85%|████████▌ | 943/1104 [21:27<03:35,  1.34s/it]

[WARN] ligne 943 | Le St. Laurent | requête échouée: Le St. Laurent, 460 St-Laurent Boulevard, Ottawa Unknown, Ottawa, ON, Canada | 400 Client Error: Bad Request for url: https://places.googleapis.com/v1/places:searchText


 99%|█████████▉| 1091/1104 [24:51<00:16,  1.28s/it]

[WARN] ligne 1091 | Olly Fresco's | requête échouée: Olly Fresco's, 50 O'Connor Street, Ottawa Unknown, Ottawa, ON, Canada | 400 Client Error: Bad Request for url: https://places.googleapis.com/v1/places:searchText


100%|██████████| 1104/1104 [25:09<00:00,  1.37s/it]


In [14]:
# -----------------------------
# Sauvegarde
# -----------------------------
out_path = Path("data/processed/ottawa_places_enriched_google_photos.csv")
df.to_csv(out_path, index=False)

print(f"Fichier enrichi sauvegardé : {out_path}")

filled = df["google_photo_url"].notna().sum()
total = len(df)
print(f"Lignes enrichies avec photo : {filled}/{total}")
print(f"Pourcentage : {filled/total:.2%}")

display(
    df.loc[df["google_photo_url"].notna(),
           ["name", "google_photo_url", "google_photo_attribution", "google_photo_match_score"]]
      .head(20)
)

Fichier enrichi sauvegardé : data\processed\ottawa_places_enriched_google_photos.csv
Lignes enrichies avec photo : 741/1104
Pourcentage : 67.12%


,name,google_photo_url,google_photo_attribution,google_photo_match_score
0,Les Suites Hotel,https://places.googleapis.com/v1/places/ChIJpX...,Les Suites Hotel Ottawa,0.8637
1,Embassy Hotel and Suites,https://places.googleapis.com/v1/places/ChIJJS...,Ottawa Embassy Hotel and Suites,0.8933
2,The Business Inn & Suites,https://places.googleapis.com/v1/places/ChIJOV...,The Business Inn & Suites,0.9429
3,Ottawa Jail Hostel,https://places.googleapis.com/v1/places/ChIJQ4...,Saintlo Ottawa Jail Hostel,0.8394
5,Hampton Inn by Hilton Ottawa,https://places.googleapis.com/v1/places/ChIJ7Q...,Hampton Inn by Hilton Ottawa,0.9779
11,Fairmont Château Laurier,https://places.googleapis.com/v1/places/ChIJRZ...,Fairmont Château Laurier,0.7970
13,Red Lobster,https://places.googleapis.com/v1/places/ChIJr5...,Kin Chan,0.9429
14,Kanata Noodle House,https://places.googleapis.com/v1/places/ChIJkW...,T.,0.8679
15,Comfort Inn Ottawa West - Kanata,https://places.googleapis.com/v1/places/ChIJXR...,Comfort Inn West,0.7531
16,Days Inn Ottawa West,https://places.googleapis.com/v1/places/ChIJQ9...,Nabil Halim,0.8024
